# Reproducibility Check

This notebook verifies that the full WNBA player valuation pipeline can be regenerated end-to-end from raw data. It does not rerun the pipeline itself — it checks that every input/output file exists, that artifacts are internally consistent, and that the final metrics are stable across reruns.

## Pipeline Run Order

To regenerate all results from scratch, notebooks must be run in this order:

| Step | Notebook | Inputs | Outputs |
|---|---|---|---|
| 1 | `preprocessing_pipeline.ipynb` | `data/raw/*.csv` | `data/processed/X_processed.csv` |
| 2 | `train_test_processed.ipynb` | `data/raw/*.csv` | `data/processed/X_train_processed.csv`, `X_test_2025_processed.csv`, `y_train.csv`, `y_test_2025.csv`, `player_lookup_*.csv` |
| 3 | `final_pipeline.ipynb` | `data/processed/*` | `artifacts/final_model.joblib`, `results/final/final_metrics.csv`, `results/final/final_predictions.csv` |



## Setup

In [1]:
import json
import hashlib
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

raw_dir          = ROOT / "data" / "raw"
processed_dir    = ROOT / "data" / "processed"
artifact_dir     = ROOT / "artifacts"
final_result_dir = ROOT / "results" / "final"
figures_dir      = final_result_dir / "figures"

PASS = "PASS"
FAIL = "FAIL"

check_log = []  # All check results accumulated here

def log(name, passed, detail=""):
    status = PASS if passed else FAIL
    check_log.append({"check": name, "status": status, "detail": detail})
    print(f"{status}  {name}" + (f" — {detail}" if detail else ""))

print("Setup OK")

Setup OK


## Check 1: Raw Data Files Exist

Verifies that all raw CSVs needed to run the pipeline from scratch are present in `data/raw/`.

In [2]:
required_raw = []
for year in range(2021, 2026):
    required_raw += [
        f"{year}_advanced.csv",
        f"{year}_per_game.csv",
        f"{year}_totals.csv",
        f"salary_{year}.csv",
        f"{year}_advanced-team.csv",
        f"{year}_wnba_standings.csv",
    ]

for fname in required_raw:
    fpath = raw_dir / fname
    log(f"raw/{fname}", fpath.exists(), "missing" if not fpath.exists() else "")

PASS  raw/2021_advanced.csv
PASS  raw/2021_per_game.csv
PASS  raw/2021_totals.csv
PASS  raw/salary_2021.csv
PASS  raw/2021_advanced-team.csv
PASS  raw/2021_wnba_standings.csv
PASS  raw/2022_advanced.csv
PASS  raw/2022_per_game.csv
PASS  raw/2022_totals.csv
PASS  raw/salary_2022.csv
PASS  raw/2022_advanced-team.csv
PASS  raw/2022_wnba_standings.csv
PASS  raw/2023_advanced.csv
PASS  raw/2023_per_game.csv
PASS  raw/2023_totals.csv
PASS  raw/salary_2023.csv
PASS  raw/2023_advanced-team.csv
PASS  raw/2023_wnba_standings.csv
PASS  raw/2024_advanced.csv
PASS  raw/2024_per_game.csv
PASS  raw/2024_totals.csv
PASS  raw/salary_2024.csv
PASS  raw/2024_advanced-team.csv
PASS  raw/2024_wnba_standings.csv
PASS  raw/2025_advanced.csv
PASS  raw/2025_per_game.csv
PASS  raw/2025_totals.csv
PASS  raw/salary_2025.csv
PASS  raw/2025_advanced-team.csv
PASS  raw/2025_wnba_standings.csv


## Check 2: Processed Artifacts Exist

Verifies that `train_test_processed.ipynb` outputs are all present in `data/processed/`.

In [3]:
required_processed = [
    "X_train_processed.csv",
    "X_test_2025_processed.csv",
    "y_train.csv",
    "y_test_2025.csv",
    "player_lookup_train.csv",
    "player_lookup_test_2025.csv",
]

for fname in required_processed:
    fpath = processed_dir / fname
    log(f"processed/{fname}", fpath.exists(), "missing" if not fpath.exists() else "")

PASS  processed/X_train_processed.csv
PASS  processed/X_test_2025_processed.csv
PASS  processed/y_train.csv
PASS  processed/y_test_2025.csv
PASS  processed/player_lookup_train.csv
PASS  processed/player_lookup_test_2025.csv


## Check 3: Final Results Artifacts Exist

Verifies that `final_pipeline.ipynb` and `final_visuals.ipynb` outputs are present.

In [4]:
required_final = [
    artifact_dir / "final_model.joblib",
    final_result_dir / "final_metrics.csv",
    final_result_dir / "final_predictions.csv",
    final_result_dir / "final_predictions_enriched.csv",
    final_result_dir / "final_model_metadata.json",
    final_result_dir / "team_cpw.csv",
    figures_dir / "predicted_vs_actual.png",
    figures_dir / "player_valuation_leaderboard.png",
    figures_dir / "mape_by_contract_group.png",
    figures_dir / "team_cpw_comparison.png",
    figures_dir / "feature_importance.png",
]

for fpath in required_final:
    log(str(fpath.relative_to(ROOT)), fpath.exists(), "missing" if not fpath.exists() else "")

PASS  artifacts/final_model.joblib
PASS  results/final/final_metrics.csv
PASS  results/final/final_predictions.csv
PASS  results/final/final_predictions_enriched.csv
PASS  results/final/final_model_metadata.json
PASS  results/final/team_cpw.csv
PASS  results/final/figures/predicted_vs_actual.png
PASS  results/final/figures/player_valuation_leaderboard.png
PASS  results/final/figures/mape_by_contract_group.png
PASS  results/final/figures/team_cpw_comparison.png
PASS  results/final/figures/feature_importance.png


## Check 4: Feature Consistency

Verifies that `X_train` and `X_test` have matching feature sets, and that the saved model was trained on the same features.

In [5]:
X_train = pd.read_csv(processed_dir / "X_train_processed.csv")
X_test  = pd.read_csv(processed_dir / "X_test_2025_processed.csv")
bundle  = joblib.load(artifact_dir / "final_model.joblib")

train_cols  = X_train.columns.tolist()
test_cols   = X_test.columns.tolist()
model_feats = bundle["feature_names"]

log("Train/test feature count match",
    len(train_cols) == len(test_cols),
    f"train={len(train_cols)}, test={len(test_cols)}")

log("Train/test feature names match",
    train_cols == test_cols,
    "" if train_cols == test_cols else str(set(train_cols) ^ set(test_cols)))

log("Model features match X_train columns",
    model_feats == train_cols,
    "" if model_feats == train_cols else str(set(model_feats) ^ set(train_cols)))

log("Feature count is 25",
    len(train_cols) == 25,
    f"got {len(train_cols)}")

PASS  Train/test feature count match — train=25, test=25
PASS  Train/test feature names match
PASS  Model features match X_train columns
PASS  Feature count is 25 — got 25


## Check 5: Metric Stability

Refits the final model from scratch using the same hyperparameters and verifies that holdout RMSE matches the saved `final_metrics.csv` within a small tolerance. A mismatch would indicate the saved model was trained on different data than what's currently in `data/processed/`.

In [6]:
y_train = pd.read_csv(processed_dir / "y_train.csv")["salary"]
y_test  = pd.read_csv(processed_dir / "y_test_2025.csv")["salary"]

saved_metrics = pd.read_csv(final_result_dir / "final_metrics.csv")
saved_rmse    = saved_metrics.loc[0, "RMSE"]
saved_mape    = saved_metrics.loc[0, "MAPE"]

# Refit from scratch with same hyperparameters
refit_model = RandomForestRegressor(
    n_estimators=400,
    min_samples_leaf=5,
    max_depth=None,
    random_state=26,
)
refit_model.fit(X_train, y_train)
refit_preds = refit_model.predict(X_test)

refit_rmse = np.sqrt(mean_squared_error(y_test, refit_preds))
refit_mape = mean_absolute_percentage_error(y_test, refit_preds)

rmse_tol = 1.0   # $1 tolerance — should be identical with same random seed
mape_tol = 0.001 # 0.1% tolerance

log("RMSE stable on refit",
    abs(refit_rmse - saved_rmse) <= rmse_tol,
    f"saved=${saved_rmse:,.2f}, refit=${refit_rmse:,.2f}, diff=${abs(refit_rmse - saved_rmse):,.2f}")

log("MAPE stable on refit",
    abs(refit_mape - saved_mape) <= mape_tol,
    f"saved={saved_mape:.4f}, refit={refit_mape:.4f}, diff={abs(refit_mape - saved_mape):.4f}")

print(f"\nRefit RMSE: ${refit_rmse:,.2f}  |  Saved RMSE: ${saved_rmse:,.2f}")

PASS  RMSE stable on refit — saved=$41,368.60, refit=$41,368.60, diff=$0.00
PASS  MAPE stable on refit — saved=1.2870, refit=1.2870, diff=0.0000

Refit RMSE: $41,368.60  |  Saved RMSE: $41,368.60


## Check 6: Hardship Contract Flag

Verifies that hardship players are correctly identified in the enriched predictions file and flagged in the visuals. This is a known structural limitation — hardship salaries are CBA-fixed and unrelated to performance, so their valuation tags (Underpaid/Overpaid) are artifacts, not real findings.

In [7]:
enriched = pd.read_csv(final_result_dir / "final_predictions_enriched.csv")

hardship_df = enriched[enriched["group"] == "hardship"]
n_hardship  = len(hardship_df)

log("Hardship players present in enriched predictions",
    n_hardship > 0,
    f"{n_hardship} hardship players in 2025 holdout")

# Hardship players showing up as underpaid/overpaid is expected but misleading
hardship_tagged = hardship_df["valuation_tag"].value_counts()
print(f"\nHardship player valuation tags (these are CBA artifacts, not real findings):")
print(hardship_tagged.to_string())

log("Hardship MAPE is high (expected structural failure)",
    hardship_df["abs_error"].mean() / hardship_df["actual"].median() > 1.0,
    f"mean abs error = ${hardship_df['abs_error'].mean():,.0f}, median actual = ${hardship_df['actual'].median():,.0f}")

PASS  Hardship players present in enriched predictions — 20 hardship players in 2025 holdout

Hardship player valuation tags (these are CBA artifacts, not real findings):
valuation_tag
Underpaid    14
Overpaid      5
Fair          1
PASS  Hardship MAPE is high (expected structural failure) — mean abs error = $20,690, median actual = $6,630


## Check 7: Row Count Consistency

Verifies that player counts are consistent across all pipeline artifacts — no rows dropped or duplicated unexpectedly.

In [8]:
y_train      = pd.read_csv(processed_dir / "y_train.csv")
y_test       = pd.read_csv(processed_dir / "y_test_2025.csv")
lookup_train = pd.read_csv(processed_dir / "player_lookup_train.csv")
lookup_test  = pd.read_csv(processed_dir / "player_lookup_test_2025.csv")
final_preds  = pd.read_csv(final_result_dir / "final_predictions.csv")

log("X_train rows match y_train",
    len(X_train) == len(y_train),
    f"X_train={len(X_train)}, y_train={len(y_train)}")

log("X_test rows match y_test",
    len(X_test) == len(y_test),
    f"X_test={len(X_test)}, y_test={len(y_test)}")

log("X_train rows match lookup_train",
    len(X_train) == len(lookup_train),
    f"X_train={len(X_train)}, lookup_train={len(lookup_train)}")

log("X_test rows match lookup_test",
    len(X_test) == len(lookup_test),
    f"X_test={len(X_test)}, lookup_test={len(lookup_test)}")

log("final_predictions rows match X_test",
    len(final_preds) == len(X_test),
    f"final_predictions={len(final_preds)}, X_test={len(X_test)}")

PASS  X_train rows match y_train — X_train=742, y_train=742
PASS  X_test rows match y_test — X_test=223, y_test=223
PASS  X_train rows match lookup_train — X_train=742, lookup_train=742
PASS  X_test rows match lookup_test — X_test=223, lookup_test=223
PASS  final_predictions rows match X_test — final_predictions=223, X_test=223


## Summary

In [9]:
summary_df = pd.DataFrame(check_log)
n_pass = (summary_df["status"] == PASS).sum()
n_fail = (summary_df["status"] == FAIL).sum()

print(f"\n{'='*50}")
print(f"REPRODUCIBILITY CHECK SUMMARY")
print(f"{'='*50}")
print(f"Total checks: {len(summary_df)}")
print(f"Passed:       {n_pass}")
print(f"Failed:       {n_fail}")
print(f"{'='*50}\n")

if n_fail > 0:
    print("Failed checks:")
    display(summary_df[summary_df["status"] == FAIL])
else:
    print("All checks passed. Pipeline is reproducible.")

display(summary_df)


REPRODUCIBILITY CHECK SUMMARY
Total checks: 60
Passed:       60
Failed:       0

All checks passed. Pipeline is reproducible.


,check,status,detail
0,raw/2021_advanced.csv,PASS,
1,raw/2021_per_game.csv,PASS,
2,raw/2021_totals.csv,PASS,
3,raw/salary_2021.csv,PASS,
4,raw/2021_advanced-team.csv,PASS,
5,raw/2021_wnba_standings.csv,PASS,
6,raw/2022_advanced.csv,PASS,
7,raw/2022_per_game.csv,PASS,
8,raw/2022_totals.csv,PASS,
9,raw/salary_2022.csv,PASS,
